<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/XGBoost2WWSpambaseHyperparameterSweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# xgboost2ww Spambase Hyperparameter Sweep (Underfit → Tuned → Mild Overfit)

## What this notebook does
- **Dataset**: UCI **Spambase** (4601 rows, 57 numeric features, binary target: spam=1, non-spam=0).
- **Loading protocol**: primary `fetch_openml("spambase", as_frame=False)` path with deterministic fallback to the UCI CSV URL if OpenML fails.
- **Data types / checks**: features cast to `float32`, labels cast to integer `{0,1}`; defensive NaN handling uses median imputation if needed (Spambase usually has no missing values).
- **Shortcut-feature toggle**: optional `DROP_SHORTCUT_FEATURES` flag can drop a small set of known dataset-collection shortcut features (`word_freq_george`, `word_freq_650`, `word_freq_857`, `word_freq_415`, `word_freq_85`). Default is **OFF** to preserve canonical benchmark behavior; turning it ON gives a stricter generalization stress test.
- **Split protocol**: stratified train/test split (`test_size=0.2`) with fixed `RNG` seed.
- **Training protocol**:
  - Internal model selection uses a **stratified train/valid split** from the training fold.
  - We use sufficiently large `num_boost_round` and `early_stopping_rounds` so training is not accidentally truncated too early.
  - `best_iteration` (effective boosting rounds) is recorded explicitly per config.
- **xgboost2ww protocol**: each config is converted with `xgboost2ww.convert(..., W="W7")` using fixed `nfolds`, `t_points`, and `random_state` for consistency.
- **WeightWatcher protocol**: run `watcher.analyze(randomize=True, detX=...)`; sweep tracks at least `alpha` and `rand_num_spikes` (detX optional for speed; enabled for key exemplars by default).
- **Sweep design rationale**:
  - **Stage A (regime sweep)** intentionally spans underfit / tuned / overfit-leaning parameter regimes.
  - **Stage B (local refinement)** perturbs the top Stage A configs to improve best-case test performance and observe whether best-performing points cluster near `alpha≈2`.
- **Expected qualitative pattern (hypothesis, not guarantee)**:
  - Underfit: often lower test accuracy and higher alpha (e.g., ~88%, alpha ~3.5).
  - Tuned: best test accuracy frequently near alpha around ~2.
  - Overfit-leaning: very high train acc, softer test acc, sometimes alpha < 2 and higher trap signatures.
- **Reproducibility**: all seeds are fixed where possible; minor nondeterminism can still occur from low-level threading / library internals.

> This notebook is designed to empirically test the HTSR/SETOL-style hypothesis on Spambase: **best generalization tends to appear near alpha≈2, while alpha<2 with trap activity can warn about instability/overfit**.


In [ ]:
# Optional install cell (useful in Colab)
# !pip -q install xgboost weightwatcher xgboost2ww scikit-learn pandas matplotlib


## Imports and global settings


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert


## Reproducibility + experiment toggles


In [ ]:
RNG = 123
np.random.seed(RNG)
TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.20
NFOLDS = 5
T_POINTS = 40
W_MATRIX = "W7"
DROP_SHORTCUT_FEATURES = False
ENABLE_OPTIONAL_FEATURE_TRANSFORMS = False
ENABLE_OPTIONAL_FEATURE_ENGINEERING = False
DET_X_SWEEP = False
DET_X_KEY_MODELS = True
SMALL_EPS = 1e-6


## Load Spambase with deterministic fallback


In [ ]:
SPAMBASE_FEATURES = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", "word_freq_our", "word_freq_over",
    "word_freq_remove", "word_freq_internet", "word_freq_order", "word_freq_mail", "word_freq_receive", "word_freq_will",
    "word_freq_people", "word_freq_report", "word_freq_addresses", "word_freq_free", "word_freq_business", "word_freq_email",
    "word_freq_you", "word_freq_credit", "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money",
    "word_freq_hp", "word_freq_hpl", "word_freq_george", "word_freq_650", "word_freq_lab", "word_freq_labs",
    "word_freq_telnet", "word_freq_857", "word_freq_data", "word_freq_415", "word_freq_85", "word_freq_technology",
    "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct", "word_freq_cs", "word_freq_meeting",
    "word_freq_original", "word_freq_project", "word_freq_re", "word_freq_edu", "word_freq_table", "word_freq_conference",
    "char_freq_;", "char_freq_(", "char_freq_[", "char_freq_!", "char_freq_$", "char_freq_#",
    "capital_run_length_average", "capital_run_length_longest", "capital_run_length_total"
]
SHORTCUT_FEATURES = ["word_freq_george", "word_freq_650", "word_freq_857", "word_freq_415", "word_freq_85"]

def load_spambase():
    try:
        ds = fetch_openml(name="spambase", version=1, as_frame=False, parser="auto")
        X = ds.data.astype(np.float32)
        y = np.asarray(ds.target)
        if y.dtype.kind in "OUS":
            y = np.where(np.isin(y.astype(str), ["1", "spam", "yes", "True"]), 1, 0)
        y = y.astype(int)
        src = "openml"
    except Exception as e:
        print(f"OpenML fetch failed ({type(e).__name__}: {e}); falling back to UCI CSV.")
        df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data", header=None)
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
        y = df.iloc[:, -1].to_numpy(dtype=int)
        src = "uci_csv"
    return X, y, src

X_raw, y_raw, data_source = load_spambase()
if np.isnan(X_raw).any():
    med = np.nanmedian(X_raw, axis=0)
    inds = np.where(np.isnan(X_raw))
    X_raw[inds] = med[inds[1]]

X_df = pd.DataFrame(X_raw, columns=SPAMBASE_FEATURES)
if DROP_SHORTCUT_FEATURES:
    X_df = X_df.drop(columns=[c for c in SHORTCUT_FEATURES if c in X_df.columns])
if ENABLE_OPTIONAL_FEATURE_TRANSFORMS:
    skew_cols = [c for c in X_df.columns if c.startswith("word_freq_") or c.startswith("char_freq_")]
    X_df[skew_cols] = np.log1p(X_df[skew_cols])
    X_df[skew_cols] = X_df[skew_cols].clip(upper=X_df[skew_cols].quantile(0.995), axis=1)
if ENABLE_OPTIONAL_FEATURE_ENGINEERING:
    X_df["ratio_bang_to_dollar"] = X_df["char_freq_!"] / (X_df["char_freq_$"] + SMALL_EPS)
    X_df["ratio_money_to_credit"] = X_df["word_freq_money"] / (X_df["word_freq_credit"] + SMALL_EPS)
    X_df["caps_x_free"] = X_df["capital_run_length_total"] * X_df["word_freq_free"]
    X_df["caps_x_you"] = X_df["capital_run_length_longest"] * X_df["word_freq_you"]

X = X_df.to_numpy(dtype=np.float32)
y = y_raw.astype(int)
assert set(np.unique(y)).issubset({0,1})


## Guardrails: shape, class balance, split sanity


In [ ]:
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y)
Xtr, Xte = X[idx_train], X[idx_test]
ytr, yte = y[idx_train], y[idx_test]
print("Data source:", data_source)
print("X shape:", X.shape, "y shape:", y.shape)
print("Class balance:", float(y.mean()))
print("Train/Test sizes:", Xtr.shape[0], Xte.shape[0])
print("NaNs in X:", bool(np.isnan(X).any()))
print("Unique labels:", np.unique(y))


## Training/evaluation helpers + xgboost2ww/WeightWatcher extraction


In [ ]:
BASE_PARAMS = {"objective":"binary:logistic","eval_metric":["logloss","auc"],"tree_method":"hist","seed":RNG}

def _extract_ww_metrics(details_df):
    alpha = np.nan; spikes = np.nan; detx_val = np.nan
    if isinstance(details_df, pd.DataFrame) and len(details_df) > 0:
        if "alpha" in details_df.columns:
            vals=pd.to_numeric(details_df["alpha"], errors="coerce"); alpha=float(vals.dropna().iloc[0]) if vals.notna().any() else np.nan
        for c in ["rand_num_spikes","num_rand_spikes","rand_spikes"]:
            if c in details_df.columns:
                vals=pd.to_numeric(details_df[c], errors="coerce"); spikes=float(vals.dropna().iloc[0]) if vals.notna().any() else np.nan; break
        for c in ["detX_val","detX","detX_num"]:
            if c in details_df.columns:
                vals=pd.to_numeric(details_df[c], errors="coerce"); detx_val=float(vals.dropna().iloc[0]) if vals.notna().any() else np.nan; break
    return alpha, spikes, detx_val

def train_eval_one(config, tag, detx=False):
    tr_idx, va_idx = train_test_split(np.arange(len(ytr)), test_size=VALID_SIZE_FROM_TRAIN, random_state=RNG, stratify=ytr)
    X_train_inner, y_train_inner = Xtr[tr_idx], ytr[tr_idx]
    X_valid_inner, y_valid_inner = Xtr[va_idx], ytr[va_idx]
    dtrain=xgb.DMatrix(X_train_inner,label=y_train_inner); dvalid=xgb.DMatrix(X_valid_inner,label=y_valid_inner)
    dtrain_full=xgb.DMatrix(Xtr,label=ytr); dtest=xgb.DMatrix(Xte,label=yte)
    params=BASE_PARAMS.copy()
    for k in ["max_depth","eta","subsample","colsample_bytree","min_child_weight","gamma","reg_alpha","reg_lambda"]: params[k]=config[k]
    n=int(config["num_boost_round"]); esr=config.get("early_stopping_rounds",None)
    model=xgb.train(params=params,dtrain=dtrain,num_boost_round=n,evals=[(dtrain,"train"),(dvalid,"valid")],early_stopping_rounds=esr,verbose_eval=False)
    br=int(getattr(model,"best_iteration",n-1)+1)
    p_train=model.predict(dtrain_full, iteration_range=(0, br)); p_test=model.predict(dtest, iteration_range=(0, br))
    row={"tag":tag,"group":config.get("group","unknown"),"max_depth":config["max_depth"],"eta":config["eta"],"subsample":config["subsample"],"colsample_bytree":config["colsample_bytree"],"min_child_weight":config["min_child_weight"],"gamma":config["gamma"],"reg_alpha":config["reg_alpha"],"reg_lambda":config["reg_lambda"],"num_boost_round":n,"early_stopping_rounds":esr,"best_round":br,"train_accuracy":accuracy_score(ytr,(p_train>=0.5).astype(int)),"test_accuracy":accuracy_score(yte,(p_test>=0.5).astype(int)),"test_auc":roc_auc_score(yte,p_test),"test_logloss":log_loss(yte,p_test)}
    converted=convert(model=model,data=Xtr,labels=ytr,W=W_MATRIX,nfolds=NFOLDS,t_points=T_POINTS,random_state=RNG,train_params=params,num_boost_round=br,multiclass="error",return_type="torch",verbose=False)
    details=ww.WeightWatcher(model=converted).analyze(randomize=True, detX=detx)
    row["alpha"], row["rand_num_spikes"], row["detX_val"] = _extract_ww_metrics(details)
    return row, model, details


## Stage A: regime sweep (underfit / tuned / overfit-leaning)


In [ ]:
stage_a_configs=[
 dict(group="underfit",max_depth=1,eta=0.30,num_boost_round=80,min_child_weight=15,gamma=5,subsample=0.60,colsample_bytree=0.60,reg_lambda=30,reg_alpha=1.0,early_stopping_rounds=20),
 dict(group="underfit",max_depth=2,eta=0.20,num_boost_round=120,min_child_weight=10,gamma=2,subsample=0.70,colsample_bytree=0.70,reg_lambda=20,reg_alpha=0.5,early_stopping_rounds=25),
 dict(group="underfit",max_depth=1,eta=0.15,num_boost_round=200,min_child_weight=20,gamma=8,subsample=0.55,colsample_bytree=0.55,reg_lambda=50,reg_alpha=1.0,early_stopping_rounds=30),
 dict(group="underfit",max_depth=3,eta=0.10,num_boost_round=180,min_child_weight=12,gamma=3,subsample=0.75,colsample_bytree=0.70,reg_lambda=15,reg_alpha=0.5,early_stopping_rounds=30),
 dict(group="tuned",max_depth=4,eta=0.08,num_boost_round=1200,min_child_weight=2,gamma=0,subsample=0.90,colsample_bytree=0.90,reg_lambda=5,reg_alpha=0.0,early_stopping_rounds=80),
 dict(group="tuned",max_depth=5,eta=0.05,num_boost_round=2000,min_child_weight=1,gamma=0,subsample=0.90,colsample_bytree=0.90,reg_lambda=2,reg_alpha=0.1,early_stopping_rounds=120),
 dict(group="tuned",max_depth=6,eta=0.05,num_boost_round=1800,min_child_weight=3,gamma=1,subsample=0.85,colsample_bytree=0.85,reg_lambda=4,reg_alpha=0.2,early_stopping_rounds=100),
 dict(group="tuned",max_depth=4,eta=0.10,num_boost_round=900,min_child_weight=2,gamma=0.5,subsample=0.95,colsample_bytree=0.90,reg_lambda=3,reg_alpha=0.0,early_stopping_rounds=60),
 dict(group="overfit",max_depth=8,eta=0.15,num_boost_round=2000,min_child_weight=1,gamma=0,subsample=1.00,colsample_bytree=1.00,reg_lambda=0.5,reg_alpha=0.0,early_stopping_rounds=None),
 dict(group="overfit",max_depth=10,eta=0.20,num_boost_round=3000,min_child_weight=1,gamma=0,subsample=1.00,colsample_bytree=1.00,reg_lambda=0.0,reg_alpha=0.0,early_stopping_rounds=None),
 dict(group="overfit",max_depth=7,eta=0.12,num_boost_round=2400,min_child_weight=1,gamma=0,subsample=1.00,colsample_bytree=1.00,reg_lambda=0.2,reg_alpha=0.0,early_stopping_rounds=None),
 dict(group="overfit",max_depth=9,eta=0.08,num_boost_round=3500,min_child_weight=1,gamma=0,subsample=1.00,colsample_bytree=1.00,reg_lambda=0.0,reg_alpha=0.0,early_stopping_rounds=300),
]
rows=[]
for i,cfg in enumerate(stage_a_configs,1):
    row,_,_=train_eval_one(cfg, tag=f"{cfg['group']}_{i:02d}", detx=DET_X_SWEEP); rows.append(row)
stage_a_df=pd.DataFrame(rows).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
stage_a_df.head(10)


## Guardrail: enforce regime spread; auto-append extremes if too narrow


In [ ]:
def extra_if_needed(df):
    add=[]; tr=df.test_accuracy.max()-df.test_accuracy.min(); gap=(df.train_accuracy-df.test_accuracy).max()
    print("test accuracy range", round(float(tr),4), "max train-test gap", round(float(gap),4))
    if tr<0.05:
        add += [dict(group="underfit",max_depth=1,eta=0.30,num_boost_round=60,min_child_weight=25,gamma=10,subsample=0.50,colsample_bytree=0.50,reg_lambda=60,reg_alpha=2.0,early_stopping_rounds=15),
                dict(group="overfit",max_depth=10,eta=0.20,num_boost_round=4000,min_child_weight=1,gamma=0,subsample=1.0,colsample_bytree=1.0,reg_lambda=0.0,reg_alpha=0.0,early_stopping_rounds=None)]
    if gap<0.02:
        add += [dict(group="overfit",max_depth=10,eta=0.15,num_boost_round=3000,min_child_weight=1,gamma=0,subsample=1.0,colsample_bytree=1.0,reg_lambda=0.0,reg_alpha=0.0,early_stopping_rounds=None)]
    return add
more=[]
for j,cfg in enumerate(extra_if_needed(stage_a_df),1):
    row,_,_=train_eval_one(cfg, tag=f"{cfg['group']}_extra_{j:02d}", detx=DET_X_SWEEP); more.append(row)
if more: stage_a_df=pd.concat([stage_a_df,pd.DataFrame(more)],ignore_index=True)
stage_a_df=stage_a_df.sort_values("test_accuracy", ascending=False).reset_index(drop=True)


## Stage B: local refinement around top Stage A configs


In [ ]:
tops=stage_a_df.head(3)
ref=[]
for _,r in tops.iterrows():
    for eta_scale,lam_scale,alp_delta,mcw_delta,ss_delta,cs_delta in [(0.85,1.2,0.0,0,+0.03,+0.03),(1.15,0.8,-0.05,-1,-0.03,-0.03),(1.00,1.0,+0.1,+1,0.0,0.0)]:
        ref.append(dict(group="refine",max_depth=int(r.max_depth),eta=float(np.clip(r.eta*eta_scale,0.03,0.25)),num_boost_round=int(max(500,r.num_boost_round)),min_child_weight=float(max(1,r.min_child_weight+mcw_delta)),gamma=float(r.gamma),subsample=float(np.clip(r.subsample+ss_delta,0.7,1.0)),colsample_bytree=float(np.clip(r.colsample_bytree+cs_delta,0.7,1.0)),reg_lambda=float(max(0,r.reg_lambda*lam_scale)),reg_alpha=float(max(0,r.reg_alpha+alp_delta)),early_stopping_rounds=int(r.early_stopping_rounds) if pd.notna(r.early_stopping_rounds) else 120))
rows=[]
for i,cfg in enumerate(ref,1):
    row,_,_=train_eval_one(cfg, tag=f"refine_{i:02d}", detx=DET_X_SWEEP); rows.append(row)
all_results_df=pd.concat([stage_a_df,pd.DataFrame(rows)], ignore_index=True).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
print("Total configs:", len(all_results_df))
all_results_df.head(12)


## Best model report + explicit xgboost2ww/WeightWatcher summary row


In [ ]:
best=all_results_df.iloc[0].to_dict()
print("best tag", best["tag"], "train", round(best["train_accuracy"],4), "test", round(best["test_accuracy"],4), "auc", round(best["test_auc"],4), "logloss", round(best["test_logloss"],4), "best_round", int(best["best_round"]))
best_cfg={k:best[k] for k in ["max_depth","eta","subsample","colsample_bytree","min_child_weight","gamma","reg_alpha","reg_lambda","num_boost_round","early_stopping_rounds"]}; best_cfg["group"]="best_model"
best_detail,_,_=train_eval_one(best_cfg, tag="best_model", detx=DET_X_KEY_MODELS)
best_summary_df=pd.DataFrame([{"model_name":"best_model","train_acc":best_detail["train_accuracy"],"test_acc":best_detail["test_accuracy"],"alpha":best_detail["alpha"],"rand_num_spikes":best_detail["rand_num_spikes"],"detX_val":best_detail["detX_val"],"best_round":best_detail["best_round"]}])
best_summary_df


## Optional detX cross-check on one underfit + one overfit exemplar


In [ ]:
ex=[]
for label,pick in [("underfit_exemplar", all_results_df[all_results_df.group.str.contains("underfit")].sort_values("test_accuracy").head(1)),("overfit_exemplar", all_results_df[all_results_df.group.str.contains("overfit")].sort_values("train_accuracy",ascending=False).head(1))]:
    if len(pick)==0: continue
    r=pick.iloc[0]
    cfg={"group":label,"max_depth":int(r.max_depth),"eta":float(r.eta),"subsample":float(r.subsample),"colsample_bytree":float(r.colsample_bytree),"min_child_weight":float(r.min_child_weight),"gamma":float(r.gamma),"reg_alpha":float(r.reg_alpha),"reg_lambda":float(r.reg_lambda),"num_boost_round":int(r.num_boost_round),"early_stopping_rounds":int(r.early_stopping_rounds) if pd.notna(r.early_stopping_rounds) else None}
    row,_,_=train_eval_one(cfg, tag=label, detx=DET_X_KEY_MODELS); ex.append(row)
pd.DataFrame(ex)


## Sweep outputs and plots (Adult-style)


In [ ]:
plot_df=all_results_df.copy()
fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].scatter(plot_df.alpha, plot_df.test_accuracy, s=60); axes[0].axvline(2.0, linestyle="--", color="red", alpha=0.8); axes[0].set_xlabel("alpha"); axes[0].set_ylabel("test accuracy"); axes[0].set_title("Test accuracy vs alpha")
axes[1].scatter(plot_df.alpha, plot_df.train_accuracy, s=60); axes[1].axvline(2.0, linestyle="--", color="red", alpha=0.8); axes[1].set_xlabel("alpha"); axes[1].set_ylabel("train accuracy"); axes[1].set_title("Train accuracy vs alpha")
axes[2].scatter(plot_df.rand_num_spikes, plot_df.test_accuracy, s=60); axes[2].set_xlabel("rand_num_spikes"); axes[2].set_ylabel("test accuracy"); axes[2].set_title("Test accuracy vs traps")
for _,r in plot_df.iterrows():
    axes[0].annotate(r.tag, (r.alpha, r.test_accuracy), fontsize=7); axes[1].annotate(r.tag, (r.alpha, r.train_accuracy), fontsize=7)
plt.tight_layout(); plt.show()


## Interpretation guide
- Look for whether strongest test models cluster near **alpha ≈ 2**.
- Watch for **alpha < 2** combined with larger train-test gap and/or higher `rand_num_spikes` (possible overfit/instability warning).
- Underfit points should generally show lower performance and often higher alpha.
- This is an empirical diagnostic notebook: the alpha/trap pattern is a **hypothesis to validate**, not a theorem.


## Save sweep results


In [ ]:
out_csv="notebooks/spambase_hyperparameter_sweep_results.csv"
all_results_df.to_csv(out_csv,index=False)
print("Saved", out_csv)
print(all_results_df[["tag","group","train_accuracy","test_accuracy","alpha","rand_num_spikes"]].head(15))
